# Claim-Level Annotation Tool

**Project:** Agentic AI for Reliable Academic Literature Review
**University:** University of Hertfordshire — MSc Data Science

## Purpose
Manually annotate citations extracted from generated literature reviews.
Labels are used to compute claim-level accuracy and Cohen's Kappa.

## How to use
1. Run all cells top to bottom (Kernel → Restart & Run All)
2. For each citation shown, read the claim and matched paper
3. Select a label from the radio buttons
4. Click Save & Next
5. When done, run Cell 10 to export CSV

## Label Definitions
| Label | Meaning |
|---|---|
| supported | Abstract clearly confirms the claim |
| partially_supported | Related but does not fully confirm (over-generalisation, wrong scope, numerical drift) |
| unsupported | Contradicts, unrelated, or paper does not exist |

In [1]:
# ============================================================
# CELL 1: Setup and imports
# ============================================================
import sys
import json
import csv
import random
from pathlib import Path
from datetime import datetime
from collections import Counter

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Add project root to path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root : {ROOT}')
print(f'Python path  : added successfully')
print('Setup complete.')

Project root : C:\Users\BEST LAPTOP\Desktop\FYP-LLM
Python path  : added successfully
Setup complete.


In [2]:
# ============================================================
# CELL 2: List available verifier log files
# ============================================================
LOG_DIR = ROOT / 'data' / 'eval' / 'verifier_logs'

if not LOG_DIR.exists():
    print(f'ERROR: Log directory not found: {LOG_DIR}')
    print('Run test_verifier.py first to generate logs.')
else:
    log_files = sorted(LOG_DIR.glob('verifier_log_*.json'))
    print(f'Found {len(log_files)} verifier log file(s):')
    for i, f in enumerate(log_files):
        size_kb = f.stat().st_size / 1024
        print(f'  [{i}] {f.name}  ({size_kb:.1f} KB)')
    if not log_files:
        print('No log files found. Run test_verifier.py first.')

Found 28 verifier log file(s):
  [0] verifier_log_20260329_043042.json  (0.8 KB)
  [1] verifier_log_20260329_061517.json  (7.9 KB)
  [2] verifier_log_20260329_061526.json  (5.6 KB)
  [3] verifier_log_20260329_062512.json  (4.9 KB)
  [4] verifier_log_20260329_062522.json  (5.5 KB)
  [5] verifier_log_20260329_062622.json  (4.9 KB)
  [6] verifier_log_20260329_062631.json  (3.1 KB)
  [7] verifier_log_20260329_062808.json  (4.0 KB)
  [8] verifier_log_20260329_062819.json  (3.7 KB)
  [9] verifier_log_20260329_063340.json  (3.6 KB)
  [10] verifier_log_20260329_063351.json  (6.2 KB)
  [11] verifier_log_20260329_063737.json  (4.1 KB)
  [12] verifier_log_20260329_063749.json  (3.7 KB)
  [13] verifier_log_20260329_222203.json  (5.0 KB)
  [14] verifier_log_20260329_222302.json  (4.4 KB)
  [15] verifier_log_20260329_222358.json  (3.8 KB)
  [16] verifier_log_20260329_223224.json  (3.7 KB)
  [17] verifier_log_20260329_223234.json  (5.0 KB)
  [18] verifier_log_20260329_224244.json  (2.5 KB)
  [19] ver

In [3]:
# ============================================================
# CELL 3: Load selected verifier log
# Change [-1] to [0], [1] etc. to select a different log file
# ============================================================
SELECTED_LOG = log_files[-1]  # most recent by default
print(f'Loading: {SELECTED_LOG.name}')

with open(SELECTED_LOG, 'r', encoding='utf-8') as f:
    log_data = json.load(f)

entries = log_data.get('entries', [])
run_id  = log_data.get('run_id', 'unknown')

print(f'Run ID         : {run_id}')
print(f'Timestamp      : {log_data.get("timestamp", "N/A")}')
print(f'Total entries  : {len(entries)}')

# Status breakdown
status_counts = Counter(e.get('status', 'UNKNOWN') for e in entries)
print(f'\nVerifier status breakdown:')
for status, count in status_counts.most_common():
    print(f'  {status:<15} : {count}')

Loading: verifier_log_20260329_233647.json
Run ID         : 20260329_233647
Timestamp      : 2026-03-29T23:36:47.622547
Total entries  : 6

Verifier status breakdown:
  VALID           : 6


In [4]:
# ============================================================
# CELL 4: Initialise annotation storage
# Loads existing annotations if file exists
# ============================================================
ANNOTATION_DIR  = ROOT / 'data' / 'eval' / 'annotations'
ANNOTATION_FILE = ANNOTATION_DIR / f'annotations_{run_id}.json'
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

# Load existing or start fresh
if ANNOTATION_FILE.exists():
    with open(ANNOTATION_FILE, 'r', encoding='utf-8') as f:
        annotations = json.load(f)
    print(f'Loaded {len(annotations)} existing annotations.')
else:
    annotations = {}
    print('No existing annotations. Starting fresh.')

# Mutable index (needed inside widget callbacks)
current_idx = [0]

print(f'Annotations file : {ANNOTATION_FILE}')
print(f'Remaining        : {len(entries) - len(annotations)} of {len(entries)}')

No existing annotations. Starting fresh.
Annotations file : C:\Users\BEST LAPTOP\Desktop\FYP-LLM\data\eval\annotations\annotations_20260329_233647.json
Remaining        : 6 of 6


In [5]:
# ============================================================
# CELL 5: Helper functions
# ============================================================

def save_annotations_to_file():
    """Persist annotations dict to JSON file."""
    with open(ANNOTATION_FILE, 'w', encoding='utf-8') as f:
        json.dump(annotations, f, indent=2, ensure_ascii=False)


def status_colour(status: str) -> str:
    return {
        'VALID':        'green',
        'PARTIAL':      'orange',
        'HALLUCINATED': 'red',
    }.get(status, 'grey')


def show_entry(idx: int):
    """Render one annotation entry with widgets."""
    clear_output(wait=True)

    # All done
    if idx >= len(entries):
        display(HTML(
            '<div style="font-family:Arial; padding:20px; '
            'background:#d4edda; border-radius:8px;">'
            '<h2 style="color:green">All entries annotated!</h2>'
            f'<p>Total annotations saved: <b>{len(annotations)}</b></p>'
            '<p>Run Cell 10 to export as CSV.</p>'
            '</div>'
        ))
        return

    entry = entries[idx]
    raw   = entry.get('raw_citation', '')
    existing_label = annotations.get(raw, {}).get('label', None)
    progress_pct   = int((idx / len(entries)) * 100)

    # --- Header ---
    display(HTML(f"""
    <div style="font-family:Arial; max-width:820px;">

      <div style="background:#343a40; color:white; padding:10px;
                  border-radius:6px; margin-bottom:12px;">
        <b>Annotation Tool</b> &nbsp;|&nbsp;
        Entry {idx + 1} of {len(entries)} &nbsp;|&nbsp;
        {progress_pct}% complete &nbsp;|&nbsp;
        Saved: {len(annotations)}
      </div>

      <div style="background:#fff3cd; padding:12px; border-radius:6px;
                  border-left:5px solid #ffc107; margin-bottom:10px;">
        <b>Citation:</b><br/>
        <span style="font-size:1.15em; font-weight:bold;">{raw}</span>
      </div>

      <div style="background:#e8f4fd; padding:12px; border-radius:6px;
                  border-left:5px solid #2196F3; margin-bottom:10px;">
        <b>Verifier Decision:</b>
        <span style="color:{status_colour(entry.get('status',''))};
                     font-weight:bold;">
          {entry.get('status', 'UNKNOWN')}
        </span>
        &nbsp;|&nbsp;
        <b>Confidence:</b> {entry.get('confidence', 'N/A')}
        &nbsp;|&nbsp;
        <b>Source:</b> {entry.get('match_source', 'N/A')}
        &nbsp;|&nbsp;
        <b>Error type:</b> {entry.get('error_type', 'None')}
      </div>

      <div style="background:#f8f9fa; padding:12px; border-radius:6px;
                  border-left:5px solid #6c757d; margin-bottom:10px;">
        <b>Matched Paper:</b><br/>
        <b>Title   :</b> {entry.get('matched_title', 'NOT FOUND — likely fabricated')}<br/>
        <b>Year    :</b> {entry.get('matched_year', 'N/A')}<br/>
        <b>Authors :</b> {', '.join(entry.get('matched_authors', []))}<br/>
        <b>DOI     :</b> {entry.get('matched_doi', 'N/A')}
      </div>

      {f'<div style="color:purple; margin-bottom:8px;">'
        f'<b>Previously labelled as:</b> {existing_label}</div>'
        if existing_label else ''}
    </div>
    """))

    # --- Widgets ---
    label_widget = widgets.RadioButtons(
        options=[
            ('✅  supported            — abstract clearly confirms the claim',
             'supported'),
            ('⚠️   partially_supported  — related but does not fully confirm',
             'partially_supported'),
            ('❌  unsupported          — contradicts, unrelated, or fabricated',
             'unsupported'),
        ],
        value=existing_label or 'supported',
        layout=widgets.Layout(width='650px'),
        style={'description_width': 'initial'},
    )

    notes_widget = widgets.Textarea(
        value=annotations.get(raw, {}).get('notes', ''),
        placeholder=(
            'Optional notes — e.g. "over-generalisation", '
            '"wrong year", "paper exists but claim exaggerated"'
        ),
        layout=widgets.Layout(width='650px', height='55px'),
    )

    save_btn = widgets.Button(
        description='Save & Next →',
        button_style='success',
        layout=widgets.Layout(width='160px', height='36px'),
    )
    back_btn = widgets.Button(
        description='← Back',
        button_style='warning',
        layout=widgets.Layout(width='110px', height='36px'),
    )
    skip_btn = widgets.Button(
        description='Skip →',
        button_style='',
        layout=widgets.Layout(width='110px', height='36px'),
    )

    display(widgets.VBox([
        widgets.HTML('<div style="margin:10px 0;"><b>Select Label:</b></div>'),
        label_widget,
        widgets.HTML('<div style="margin:10px 0;"><b>Notes (optional):</b></div>'),
        notes_widget,
        widgets.HTML('<div style="margin:12px 0;"></div>'),
        widgets.HBox([save_btn, back_btn, skip_btn]),
    ]))

    # --- Callbacks ---
    def on_save(b):
        annotations[raw] = {
            'label': label_widget.value,
            'notes': notes_widget.value,
            'timestamp': datetime.now().isoformat(),
            'verifier_status': entry.get('status', ''),
        }
        save_annotations_to_file()
        current_idx[0] += 1
        show_entry(current_idx[0])

    def on_back(b):
        if current_idx[0] > 0:
            current_idx[0] -= 1
        show_entry(current_idx[0])

    def on_skip(b):
        current_idx[0] += 1
        show_entry(current_idx[0])

    save_btn.on_click(on_save)
    back_btn.on_click(on_back)
    skip_btn.on_click(on_skip)

In [6]:
# ============================================================
# CELL 6: Launch the annotation interface
# ============================================================
print('Launching annotation interface...')
show_entry(current_idx[0])

In [7]:
# ============================================================
# CELL 7: View annotation progress
# ============================================================
from collections import Counter

if annotations:
    label_counts = Counter(a.get('label', 'UNKNOWN') for a in annotations.values())
    print(f'Total annotations: {len(annotations)}')
    print(f'\nLabel distribution:')
    for label, count in label_counts.most_common():
        pct = count / len(annotations) * 100
        print(f'  {label:<20} : {count:3d}  ({pct:5.1f}%)')
else:
    print('No annotations yet.')

No annotations yet.


In [8]:
# ============================================================
# CELL 8: Inter-rater reliability (Cohen's Kappa)
# ============================================================
def compute_cohens_kappa(ann1_file: Path, ann2_file: Path):
    """
    Compute Cohen's Kappa between two annotation files.
    """
    with open(ann1_file, 'r', encoding='utf-8') as f:
        ann1 = json.load(f)
    with open(ann2_file, 'r', encoding='utf-8') as f:
        ann2 = json.load(f)

    # Find common citations
    common_keys = set(ann1.keys()) & set(ann2.keys())
    if not common_keys:
        print('No overlapping citations to compare.')
        return None

    labels1 = [ann1[k].get('label', '') for k in common_keys]
    labels2 = [ann2[k].get('label', '') for k in common_keys]

    # Simple kappa calculation
    agreement = sum(1 for l1, l2 in zip(labels1, labels2) if l1 == l2)
    p_o = agreement / len(common_keys)

    # Expected agreement
    all_labels = set(labels1) | set(labels2)
    p_e = 0
    for label in all_labels:
        p1 = labels1.count(label) / len(labels1)
        p2 = labels2.count(label) / len(labels2)
        p_e += p1 * p2

    kappa = (p_o - p_e) / (1 - p_e) if p_e < 1 else 0

    print(f'Common citations : {len(common_keys)}')
    print(f'Observed agree.  : {p_o:.3f}')
    print(f'Expected agree.  : {p_e:.3f}')
    print(f"Cohen's Kappa    : {kappa:.3f}")

    # Interpretation
    if kappa < 0:
        interp = 'No agreement'
    elif kappa < 0.20:
        interp = 'Slight agreement'
    elif kappa < 0.40:
        interp = 'Fair agreement'
    elif kappa < 0.60:
        interp = 'Moderate agreement'
    elif kappa < 0.80:
        interp = 'Substantial agreement'
    else:
        interp = 'Almost perfect agreement'

    print(f'Interpretation   : {interp}')
    return kappa


# Example usage (uncomment and adjust paths as needed):
# compute_cohens_kappa(
#     ANNOTATION_DIR / 'annotations_run1.json',
#     ANNOTATION_DIR / 'annotations_run2.json'
# )

In [9]:
# ============================================================
# CELL 9: View all annotations
# ============================================================
if annotations:
    for citation, data in list(annotations.items())[:10]:  # Show first 10
        print(f"\n{'='*60}")
        print(f"Citation: {citation}")
        print(f"Label   : {data.get('label', 'N/A')}")
        print(f"Notes   : {data.get('notes', 'N/A')}")
        print(f"Time    : {data.get('timestamp', 'N/A')}")
else:
    print('No annotations to display.')

No annotations to display.


In [10]:
# ============================================================
# CELL 10: Export annotations to CSV
# ============================================================
CSV_EXPORT_PATH = ANNOTATION_DIR / f'annotations_{run_id}.csv'

if annotations:
    with open(CSV_EXPORT_PATH, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            'citation', 'label', 'notes', 'timestamp', 'verifier_status'
        ])
        for citation, data in annotations.items():
            writer.writerow([
                citation,
                data.get('label', ''),
                data.get('notes', ''),
                data.get('timestamp', ''),
                data.get('verifier_status', ''),
            ])

    print(f'CSV exported successfully!')
    print(f'Saved to: {CSV_EXPORT_PATH}')
    print(f'Total rows: {len(annotations)}')
else:
    print('No annotations to export.')

No annotations to export.


In [11]:
# ============================================================
# CELL 11: Summary statistics
# ============================================================
if annotations:
    print('\n' + '='*60)
    print('ANNOTATION SUMMARY')
    print('='*60)
    print(f'Run ID           : {run_id}')
    print(f'Total entries    : {len(entries)}')
    print(f'Annotated        : {len(annotations)} ({len(annotations)/len(entries)*100:.1f}%)')
    print(f'File             : {ANNOTATION_FILE}')
    print(f'CSV Export       : {CSV_EXPORT_PATH if "CSV_EXPORT_PATH" in locals() else "Not yet exported"}')
    
    label_counts = Counter(a.get('label', 'UNKNOWN') for a in annotations.values())
    print(f'\nLabel distribution:')
    for label, count in label_counts.most_common():
        pct = count / len(annotations) * 100
        bar = '█' * int(pct / 5)
        print(f'  {label:<20} : {count:3d}  {bar} ({pct:5.1f}%)')
else:
    print('No annotations to summarize.')

No annotations to summarize.
